# Real-data kNN classification and regression

**Lecture 13 · Notebook 01 · CMOR 438 / INDE 577**

This laboratory uses real analytical-chemistry and clinical observations from the versioned course database. The goal is not to celebrate a score. It is to build, tune, compare, and diagnose two leakage-resistant local models.

## How to use this notebook

Run core cells in order. The classification route uses Italian wine chemistry to predict cultivar; the regression route uses baseline clinical measurements to predict one-year disease progression. Treat both as teaching evidence with documented limitations—not as deployable scientific or medical systems.

## Learning objectives

You will inspect database provenance, separate train/validation/test roles, fit scaling only on training data, choose $k$ without touching test outcomes, compare against baselines and trusted implementations, visualize errors, and measure prediction latency.

## Why this matters

An algorithm is only one component of a prediction claim. Real work begins with observational units, provenance, missingness, units, leakage boundaries, and the consequences of errors. The same kNN mechanics can support classification and regression, but their metrics, baselines, and failure analyses differ.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import find_project_root

PROJECT_ROOT = find_project_root()
EXPECTED_ENVIRONMENT = PROJECT_ROOT / ".venv"

# A notebook kernel is a long-running Python process.  Its executable—not the
# terminal prompt—determines which packages this notebook imports.
print("Python version:    ", sys.version.split()[0])
print("Python executable: ", sys.executable)
print("Active prefix:     ", sys.prefix)
print("Repository root:   ", PROJECT_ROOT)
print("Using project .venv:", Path(sys.prefix).resolve() == EXPECTED_ENVIRONMENT.resolve())

if Path(sys.prefix).resolve() != EXPECTED_ENVIRONMENT.resolve():
    raise RuntimeError(
        "This notebook is not using the repository .venv. In VS Code choose "
        "Kernel > Select Another Kernel > Jupyter Kernel > Rice DSM. If it is "
        "missing, close the notebook, run `uv sync --locked` and "
        "`uv run python scripts/setup_course.py` in the repository terminal, "
        "then reopen the notebook."
    )

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
import sqlite3

from rice_dsm.paths import course_database_path

database_path = course_database_path()
database_uri = f"{database_path.as_uri()}?mode=ro"

# Read catalog metadata before selecting a table.  A table name alone does not
# tell us what a row means, where it came from, or what claim it can support.
with sqlite3.connect(database_uri, uri=True) as connection:
    catalog = pd.read_sql_query(
        """SELECT dataset_id, title, domain, task, observational_unit,
                  target_column, target_description, row_count, feature_count,
                  source_url, source_note, license_note
           FROM dataset_catalog
           WHERE dataset_id IN ('wine_recognition', 'diabetes')""",
        connection,
    )

catalog.T

## 1. Classification question: can chemistry recover cultivar labels?

One row is one wine sample; thirteen numerical features are chemical measurements; the target is one of three cultivar codes. This is a multiclass classification exercise, not a claim that cultivar is the only cause of chemical differences. We will reserve the test set before exploration that uses target labels.

In [ ]:
with sqlite3.connect(database_uri, uri=True) as connection:
    wine = pd.read_sql_query(
        "SELECT * FROM wine_observations ORDER BY observation_id",
        connection,
    )

print("shape:", wine.shape)
display(wine.head())
display(wine.dtypes.to_frame("dtype").T)
display(wine.isna().sum().to_frame("missing").T)
display(wine["cultivar_label"].value_counts().sort_index().to_frame("count"))

assert wine["observation_id"].is_unique
assert wine.isna().sum().sum() == 0
assert wine["cultivar_code"].nunique() == 3

In [ ]:
# The plot asks whether class separation and feature scale differ in two visible dimensions.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.scatterplot(data=wine, x="alcohol", y="color_intensity",
                hue="cultivar_label", style="cultivar_label", s=75, ax=axes[0])
axes[0].set(title="Cultivars overlap in raw chemical coordinates",
            xlabel="Alcohol concentration", ylabel="Color intensity")

scale_summary = wine[["alcohol", "magnesium", "proline", "color_intensity"]].agg(["mean", "std"]).T
scale_summary["std"].sort_values().plot.barh(ax=axes[1], color="#00205B")
axes[1].set(title="Raw feature scales differ sharply", xlabel="Sample standard deviation")
plt.show()

### Split roles before fitting

- **Training:** fit the scaler and store kNN observations.
- **Validation:** choose $k$ and weighting.
- **Test:** estimate final selected-pipeline performance once.

The split is stratified so each subset retains all cultivar classes. The scaler learns no target values, but fitting it on validation or test features still leaks held-out distribution information.

In [ ]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from rice_dsm.ml.neighbors import (
    KNearestNeighborsClassifier,
    KNearestNeighborsRegressor,
)

feature_names = [
    column for column in wine.columns
    if column not in {"observation_id", "cultivar_code", "cultivar_label"}
]
X = wine[feature_names].to_numpy(dtype=float)
y = wine["cultivar_label"].to_numpy()

# Hold out test first; validation is then carved only from the development set.
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.20, random_state=438, stratify=y,
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.25,
    random_state=438, stratify=y_development,
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

print("train / validation / test:", len(X_train), len(X_validation), len(X_test))
assert len(X_train) + len(X_validation) + len(X_test) == len(X)

In [ ]:
# Validation compares declared candidates; the test labels remain untouched.
classification_candidates = []
for weights in ("uniform", "distance"):
    for k in range(1, 22, 2):
        candidate = KNearestNeighborsClassifier(k, weights).fit(X_train_scaled, y_train)
        validation_accuracy = accuracy_score(
            y_validation, candidate.predict(X_validation_scaled)
        )
        classification_candidates.append({
            "k": k,
            "weights": weights,
            "validation_accuracy": validation_accuracy,
        })

classification_results = pd.DataFrame(classification_candidates)
best_row = classification_results.sort_values(
    ["validation_accuracy", "k"], ascending=[False, True]
).iloc[0]
best_k = int(best_row["k"])
best_weights = str(best_row["weights"])

display(classification_results.sort_values("validation_accuracy", ascending=False).head(8))
print("selected:", {"k": best_k, "weights": best_weights})

In [ ]:
fig, axis = plt.subplots(figsize=(9, 5))
sns.lineplot(data=classification_results, x="k", y="validation_accuracy",
             hue="weights", marker="o", ax=axis)
axis.axvline(best_k, color="0.3", linestyle="--", label="selected k")
axis.set(title="Validation—not test—selects neighborhood size",
         ylabel="Validation accuracy", xticks=range(1, 22, 2))
plt.show()

### Worked example: freeze, refit, evaluate once

After selection, we may combine training and validation, refit a fresh scaler on that combined development data, and fit the frozen kNN configuration. Only then do we reveal the test labels. We report a majority-class baseline, accuracy, and class-specific confusion—not accuracy alone.

In [ ]:
from sklearn.dummy import DummyClassifier

final_scaler = StandardScaler().fit(X_development)
X_development_scaled = final_scaler.transform(X_development)
X_test_final_scaled = final_scaler.transform(X_test)

final_classifier = KNearestNeighborsClassifier(best_k, best_weights).fit(
    X_development_scaled, y_development
)
test_predictions = final_classifier.predict(X_test_final_scaled)
baseline_predictions = DummyClassifier(strategy="most_frequent").fit(
    X_development, y_development
).predict(X_test)

print(f"majority baseline accuracy: {accuracy_score(y_test, baseline_predictions):.3f}")
print(f"selected kNN test accuracy: {accuracy_score(y_test, test_predictions):.3f}")

ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, cmap="Blues")
plt.title("Held-out cultivar confusion matrix")
plt.show()

## 2. Regression question: can baseline measurements predict progression?

One row is one patient. The target is a quantitative disease-progression measure one year after baseline. Feature meanings are incompletely documented, the sample is small, and this exercise does not establish clinical utility or causal effects. We compare against the training mean and report MAE and RMSE in target units.

In [ ]:
with sqlite3.connect(database_uri, uri=True) as connection:
    diabetes = pd.read_sql_query(
        "SELECT * FROM diabetes_observations ORDER BY observation_id",
        connection,
    )

display(diabetes.head())
display(diabetes.describe().T)
assert diabetes["observation_id"].is_unique
assert diabetes.isna().sum().sum() == 0

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Preserve the same development/test boundary used above, then let validation
# choose k and weighting under regression metrics rather than accuracy.

regression_features = [
    column for column in diabetes.columns
    if column not in {"observation_id", "disease_progression"}
]
X_reg = diabetes[regression_features].to_numpy(dtype=float)
y_reg = diabetes["disease_progression"].to_numpy(dtype=float)

X_reg_dev, X_reg_test, y_reg_dev, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=438,
)
X_reg_train, X_reg_validation, y_reg_train, y_reg_validation = train_test_split(
    X_reg_dev, y_reg_dev, test_size=0.25, random_state=438,
)
reg_scaler = StandardScaler().fit(X_reg_train)
X_reg_train_scaled = reg_scaler.transform(X_reg_train)
X_reg_validation_scaled = reg_scaler.transform(X_reg_validation)

regression_candidates = []
for weights in ("uniform", "distance"):
    for k in range(1, 32, 2):
        candidate = KNearestNeighborsRegressor(k, weights).fit(
            X_reg_train_scaled, y_reg_train
        )
        predictions = candidate.predict(X_reg_validation_scaled)
        regression_candidates.append({
            "k": k,
            "weights": weights,
            "validation_mae": mean_absolute_error(y_reg_validation, predictions),
            "validation_rmse": mean_squared_error(
                y_reg_validation, predictions
            ) ** 0.5,
        })

regression_results = pd.DataFrame(regression_candidates)
best_regression = regression_results.sort_values(
    ["validation_mae", "k"]
).iloc[0]
display(regression_results.sort_values("validation_mae").head(8))

In [ ]:
selected_reg_k = int(best_regression["k"])
selected_reg_weights = str(best_regression["weights"])
final_reg_scaler = StandardScaler().fit(X_reg_dev)
final_regressor = KNearestNeighborsRegressor(
    selected_reg_k, selected_reg_weights
).fit(final_reg_scaler.transform(X_reg_dev), y_reg_dev)

regression_test_predictions = final_regressor.predict(
    final_reg_scaler.transform(X_reg_test)
)
baseline = np.full_like(y_reg_test, y_reg_dev.mean())

regression_evidence = pd.DataFrame({
    "model": ["training-mean baseline", "selected kNN"],
    "MAE": [
        mean_absolute_error(y_reg_test, baseline),
        mean_absolute_error(y_reg_test, regression_test_predictions),
    ],
    "RMSE": [
        mean_squared_error(y_reg_test, baseline) ** 0.5,
        mean_squared_error(y_reg_test, regression_test_predictions) ** 0.5,
    ],
})
regression_evidence

In [ ]:
residuals = y_reg_test - regression_test_predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
axes[0].scatter(regression_test_predictions, y_reg_test, alpha=0.75, color="#00205B")
limits = [min(y_reg_test.min(), regression_test_predictions.min()),
          max(y_reg_test.max(), regression_test_predictions.max())]
axes[0].plot(limits, limits, "--", color="#C8102E")
axes[0].set(xlabel="Predicted progression", ylabel="Observed progression",
            title="Predictions versus outcomes")
axes[1].scatter(regression_test_predictions, residuals, alpha=0.75, color="#00205B")
axes[1].axhline(0, linestyle="--", color="#C8102E")
axes[1].set(xlabel="Predicted progression", ylabel="Observed − predicted",
            title="Residual structure and extreme errors")
plt.show()

## 3. Trusted-library differential comparison

Student-authored code should be tested against analytically known cases first. A mature library then serves as a differential measuring instrument. Agreement is expected only when preprocessing, metric, $k$, weights, and tie behavior match. Library agreement cannot prove the model is scientifically appropriate.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as SklearnKNNClassifier

reference = SklearnKNNClassifier(
    n_neighbors=best_k,
    weights=best_weights,
    metric="euclidean",
).fit(X_development_scaled, y_development)
reference_predictions = reference.predict(X_test_final_scaled)

agreement = np.mean(reference_predictions == test_predictions)
print(f"prediction agreement with scikit-learn: {agreement:.3f}")
assert agreement == 1.0

## 4. Engineering evidence: latency, memory, and support

Brute-force prediction compares each query with every stored training row: approximately $O(n_{train}d)$ time per query and $O(n_{train}d)$ fitted storage. Measure batch latency rather than extrapolating from one call. Also inspect nearest-neighbor distances: a prediction far from all training observations is an extrapolation warning even if the API returns a class.

In [ ]:
from time import perf_counter

batch_sizes = [1, 8, len(X_test_final_scaled)]
latency_rows = []
for batch_size in batch_sizes:
    batch = X_test_final_scaled[:batch_size]
    repeats = 200 if batch_size == 1 else 40
    start = perf_counter()
    for _ in range(repeats):
        final_classifier.predict(batch)
    elapsed = perf_counter() - start
    latency_rows.append({
        "batch_size": batch_size,
        "milliseconds_per_batch": 1_000 * elapsed / repeats,
        "microseconds_per_observation": 1_000_000 * elapsed / (repeats * batch_size),
    })

latency = pd.DataFrame(latency_rows)
display(latency)
print("stored feature bytes:", final_classifier.features_.nbytes)

## Professional practice and common failure modes

- Do not tune on test outcomes.
- Store the scaler with the estimator; raw-feature prediction against scaled training rows is invalid.
- Preserve feature order and units as an input schema.
- A good average metric can hide one harmed class or extreme regression errors.
- Fast fitting does not imply fast serving.
- Distance-weighted votes are not calibrated probabilities.

**Common failure:** adding many unrelated features does not let kNN “ignore” them. Noise coordinates change distances and can destroy neighborhoods—the curse of dimensionality.

## Debugging clinic

If your result differs from this notebook, compare in this order: database digest and row count; split seeds; target column; feature order; scaler fit rows; $k$ and weight mode; then library version. Printing only the final score discards the evidence needed to locate the difference.

## Guided practice

1. Repeat classification without scaling. Which neighbor identities and validation scores change?
2. Add five permuted noise features to the wine data. Plot validation accuracy and nearest/farthest distance ratio.
3. For the largest diabetes residual, inspect its neighbor distances and targets. Is the error surprising locally?

In [ ]:
# PRACTICE: compare the local target distribution for the largest held-out error.
largest_error_index = int(np.argmax(np.abs(residuals)))
query = final_reg_scaler.transform(X_reg_test[[largest_error_index]])
neighbor_distances, neighbor_indices = final_regressor.kneighbors(query)

local_evidence = pd.DataFrame({
    "distance": neighbor_distances[0],
    "training_target": y_reg_dev[neighbor_indices[0]],
}).sort_values("distance")
display(local_evidence)

assert len(local_evidence) == selected_reg_k
assert local_evidence["distance"].is_monotonic_increasing

## Independent practice

Propose one domain-aware distance for either dataset. Define its units, invariances, missing-data behavior, and a test where it changes a neighbor identity for a defensible reason. Do not implement it until the contract is reviewable.

## Extension: repeated validation

A single split can make the selected $k$ unstable in small data. Use repeated stratified cross-validation for classification or repeated K-fold validation for regression **inside the development data**, retain the untouched outer test set, and graph the distribution of metrics for each $k$. Report selection uncertainty rather than only the winning mean.

## Retrieval practice

1. Why are train, validation, and test three different roles?
2. Why can a scaler leak without reading the target?
3. What does a confusion matrix reveal that accuracy hides?
4. What state makes a kNN artifact large?
5. What evidence would warn that a query is outside training support?

## Takeaway

The same local search supports classification and regression, but professional evidence depends on the task. Real-data kNN requires provenance, a fitted transformation boundary, validation-only selection, baselines, task-appropriate errors, failure analysis, trusted differential checks, and resource measurements.

## Further reading

- [scikit-learn nearest-neighbor user guide](https://scikit-learn.org/stable/modules/neighbors.html)
- [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)
- [Python sqlite3 documentation](https://docs.python.org/3/library/sqlite3.html)